<a href="https://colab.research.google.com/github/Sagnik-Chowdhury/Federated-Learning-1/blob/Sourit/Internship_Fed_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Target Model Training & Serialization
## Federated Learning Vulnerability Analysis: Modality Comparison

**Objective:**
This notebook simulates the local client training phase in a standard Federated Learning (FL) pipeline. Instead of training a single model, we are training two separate models on distinct data modalities:
1. **Integrated/Dense Data:** High-dimensional images (MNIST).
2. **Scattered/Tabular Data:** Low-dimensional, heterogeneous features (Breast Cancer Dataset).

**Methodology:**
Both models will be trained locally. Once convergence is reached, we serialize and save their `state_dicts` (the network weights). In a real FL environment, these weights would be transmitted to a central server. In our experiment, we will intercept these saved weights in Notebook 2 to execute a Model Inversion (Feature Reconstruction) attack, comparing how easily private data can be extracted from different data structures.

## Libraries and Mounting to Drive

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Architecture

In [9]:
# For Integrated Data (MNIST Images)
class FourLayerNet(nn.Module):
    def __init__(self):
        super(FourLayerNet, self).__init__()
        # Maps 784 pixels down to 10 digit classes
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28 * 28) # Flatten the 28x28 image grid into a 1D array
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.fc4(x)

In [10]:
# For Scattered Data (Tabular Features)
class TabularNet(nn.Module):
    def __init__(self):
        super(TabularNet, self).__init__()
        # Maps 30 medical features down to 2 diagnosis classes (Malignant/Benign)
        self.fc1 = nn.Linear(30, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

## Data Preparation

Data processing differs significantly between modalities:
* **MNIST** simply requires normalization to center the pixel values around zero.
* **Breast Cancer (Tabular)** requires rigorous standard scaling. Medical features operate on wildly different scales (e.g., perimeter vs. smoothness). `StandardScaler` ensures no single feature dominates the gradients during training, which is crucial for both convergence and the subsequent inversion attack.

In [11]:
print("Preparing MNIST Dataset")

# Normalize pixels to a range of [-1, 1] for stable gradients
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
mnist_train = datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)

print("Data preparation complete")

Preparing MNIST Dataset
Data preparation complete


In [12]:
print("Preparing Breast Cancer (Tabular) Dataset")
data = load_breast_cancer()

# Initialize the StandardScaler
scaler = StandardScaler()
# Fit and transform the tabular data to have mean=0 and variance=1
X_scaled = scaler.fit_transform(data.data)

# Convert numpy arrays to PyTorch tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(data.target, dtype=torch.long)

# Wrap the tabular tensors in a DataLoader so we can iterate in batches
tabular_dataset = TensorDataset(X_tensor, y_tensor)
tabular_loader = DataLoader(tabular_dataset, batch_size=32, shuffle=True)

print("Data preparation complete")

Preparing Breast Cancer (Tabular) Dataset
Data preparation complete


## Local Training Simulation

We now simulate the local client training for 5 epochs. In a real-world scenario, this computation happens on the edge device (e.g., a hospital's local server or a user's mobile phone).

In [13]:
epochs = 5

In [15]:
# Train the MNIST Model

mnist_model = FourLayerNet()
criterion_mnist = nn.CrossEntropyLoss()
optimizer_mnist = optim.Adam(mnist_model.parameters(), lr=0.001)

print("Starting MNIST Model Training")
for epoch in range(epochs):
    for images, labels in mnist_loader:
        optimizer_mnist.zero_grad()
        outputs = mnist_model(images)
        loss = criterion_mnist(outputs, labels)
        loss.backward()
        optimizer_mnist.step()
    print(f"MNIST Epoch {epoch+1}/{epochs} completed. Loss: {loss.item():.4f}")

Starting MNIST Model Training
MNIST Epoch 1/5 completed. Loss: 0.0998
MNIST Epoch 2/5 completed. Loss: 0.1866
MNIST Epoch 3/5 completed. Loss: 0.1644
MNIST Epoch 4/5 completed. Loss: 0.0091
MNIST Epoch 5/5 completed. Loss: 0.0063


In [16]:
# Train the Tabular Model

tabular_model = TabularNet()
criterion_tab = nn.CrossEntropyLoss()
# Tabular networks often benefit from a slightly higher learning rate due to shallower depth
optimizer_tab = optim.Adam(tabular_model.parameters(), lr=0.01)

print("\nStarting Tabular Model Training")
for epoch in range(epochs):
    for features, labels in tabular_loader:
        optimizer_tab.zero_grad()
        outputs = tabular_model(features)
        loss = criterion_tab(outputs, labels)
        loss.backward()
        optimizer_tab.step()
    print(f"Tabular Epoch {epoch+1}/{epochs} completed. Loss: {loss.item():.4f}")


Starting Tabular Model Training
Tabular Epoch 1/5 completed. Loss: 0.0077
Tabular Epoch 2/5 completed. Loss: 0.0045
Tabular Epoch 3/5 completed. Loss: 0.0113
Tabular Epoch 4/5 completed. Loss: 0.0501
Tabular Epoch 5/5 completed. Loss: 0.0107


## Model Serialization & Storage

The final step is to save the highly-optimized parameter weights. By writing these `state_dicts` to disk, we simulate the transmission payload sent from the edge client to the central aggregating server. Notebook 2 will intercept these specific files.

In [17]:
# Save the MNIST weights
mnist_path = '/content/drive/MyDrive/fedavg_mnist_weights.pth'
torch.save(mnist_model.state_dict(), mnist_path)
print(f"\nMNIST Model weights securely saved to: {mnist_path}")

# Save the Tabular weights
tabular_path = '/content/drive/MyDrive/fedavg_tabular_weights.pth'
torch.save(tabular_model.state_dict(), tabular_path)
print(f"Tabular Model weights securely saved to: {tabular_path}")


MNIST Model weights securely saved to: /content/drive/MyDrive/fedavg_mnist_weights.pth
Tabular Model weights securely saved to: /content/drive/MyDrive/fedavg_tabular_weights.pth
